# Audiovisual Cortical Mapping Pipeline

Runs the three-script pipeline sequentially:

| Script | What it does | Key outputs |
|--------|-------------|-------------|
| `01_extract_geometry.py` | Glasser → A1/V1 masks → sphere LBOEs | `sub_a1.pkl`, `sub_v1.pkl` |
| `02_prep_hcp_timeseries.py` | MAT → z-score → project onto LBOEs | `X_train/test.npy`, `Y_train/test.npy` |
| `03_fit_banded_ridge.py` | Himalaya banded ridge → variance partition | `R2_*.npy`, `*.dscalar.nii` |

**Kernel:** `vicsompy_av`  
**Estimated wall-clock:** ~2 h (dominated by Script 1 LBOE computation and Script 3 fitting)

In [1]:
import importlib.util
import sys
import os
import time
import logging
import numpy as np

# ── point Python at the scripts directory ─────────────────────────────────────
SCRIPT_DIR = os.path.dirname(os.path.abspath("run_pipeline.ipynb"))
if SCRIPT_DIR not in sys.path:
    sys.path.insert(0, SCRIPT_DIR)

# ── configure logging so it shows up in notebook cells ────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,           # re-configure if already set by an imported module
    handlers=[logging.StreamHandler(sys.stdout)],
)

def load_script(name):
    """Import a pipeline script as a module by file path."""
    path = os.path.join(SCRIPT_DIR, name)
    spec = importlib.util.spec_from_file_location(name.replace(".py", ""), path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

def banner(title):
    bar = "═" * 60
    print(f"\n{bar}")
    print(f"  {title}")
    print(bar)

print("Notebook initialised.")
print(f"  Script dir : {SCRIPT_DIR}")
print(f"  Python     : {sys.version.split()[0]}")

Notebook initialised.
  Script dir : /home/amin/Research/Representation/Movie/movie_watching/vicsompy_audiovisual
  Python     : 3.10.20


---
## Script 1 — Extract Geometry & LBOEs

- Loads Glasser GIFTI parcellations → boolean masks for bilateral A1 and V1
- Loads HCP sphere surfaces directly via nibabel (bypasses pycortex DB)
- Runs Laplace-Beltrami decomposition → 200 eigenfunctions per hemisphere
- Caches results as `sub_a1.pkl` / `sub_v1.pkl` (re-run loads from cache)

> **Note:** First run takes ~5–15 min per ROI (sparse eigendecomposition). Subsequent runs load from cache instantly.

In [2]:
banner("SCRIPT 1 — Extract Geometry & LBOEs")
t0 = time.time()

s1 = load_script("01_extract_geometry.py")
s1.main()

elapsed = time.time() - t0
print(f"\n✓ Script 1 complete in {elapsed/60:.1f} min")


════════════════════════════════════════════════════════════
  SCRIPT 1 — Extract Geometry & LBOEs
════════════════════════════════════════════════════════════
11:30:21  INFO  Note: NumExpr detected 32 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
11:30:21  INFO  NumExpr defaulting to 16 threads.
11:30:21  INFO  ============================================================
11:30:21  INFO  Script 1 — Extract geometry & LBOEs
11:30:21  INFO    Pycortex subject : hcp_999999_draw_NH
11:30:21  INFO    Surface type     : sphere (HCP GIFTI, vicsompy default)
11:30:21  INFO    LBOEs per hem.   : 200
11:30:21  INFO  ============================================================
11:30:21  INFO  Loading Glasser GIFTI parcellations …
11:30:21  INFO    Left : 32492 vertices, 181 unique labels
11:30:21  INFO    Right: 32492 vertices, 181 unique labels
11:30:21  INFO    A1 Left: 77 vertices
11:30:21  INFO    A1 Right: 66 vertices
11:30:21  INFO    V1 Left: 831 vertices
11:30:2

In [3]:
# ── Quick sanity check ────────────────────────────────────────────────────────
import pickle, os

CACHE_DIR = s1.CACHE_DIR
for roi in ["a1", "v1"]:
    pkl = os.path.join(CACHE_DIR, f"sub_{roi}.pkl")
    with open(pkl, "rb") as fh:
        sub = pickle.load(fh)
    # Backward-compat: old pickles may lack n_lboe attribute
    if not hasattr(sub, "n_lboe"):
        sub.n_lboe = sub.L_eigenvectors.shape[1]
    n_lboe = sub.n_lboe
    print(f"sub_{roi}:")
    print(f"  subsurface_verts_L : {len(sub.subsurface_verts_L)} verts   "
          f"(indices in [0, 32491])")
    print(f"  subsurface_verts_R : {len(sub.subsurface_verts_R)} verts   "
          f"(indices in [32492, 64983])")
    print(f"  L_eigenvectors     : {sub.L_eigenvectors.shape}  dtype={sub.L_eigenvectors.dtype}")
    print(f"  R_eigenvectors     : {sub.R_eigenvectors.shape}")
    print(f"  n_lboe (actual)    : {n_lboe}  (target max: {s1.N_LBOE})")
    # Both hemispheres must have the same eigenfunction count
    assert sub.L_eigenvectors.shape[1] == sub.R_eigenvectors.shape[1], \
        "L/R LBOE count mismatch within ROI!"
    # Actual count must not exceed the configured target
    assert n_lboe <= s1.N_LBOE, \
        f"n_lboe={n_lboe} exceeds N_LBOE target={s1.N_LBOE}!"
    # Right hemisphere vertex indices must carry the +32492 offset
    assert sub.subsurface_verts_R.min() >= 32492, "R verts offset missing!"
    print(f"  ✓ assertions passed\n")

sub_a1:
  subsurface_verts_L : 77 verts   (indices in [0, 32491])
  subsurface_verts_R : 66 verts   (indices in [32492, 64983])
  L_eigenvectors     : (77, 64)  dtype=complex128
  R_eigenvectors     : (66, 64)
  n_lboe (actual)    : 64  (target max: 200)
  ✓ assertions passed

sub_v1:
  subsurface_verts_L : 831 verts   (indices in [0, 32491])
  subsurface_verts_R : 787 verts   (indices in [32492, 64983])
  L_eigenvectors     : (831, 200)  dtype=complex128
  R_eigenvectors     : (787, 200)
  n_lboe (actual)    : 200  (target max: 200)
  ✓ assertions passed



---
## Script 2 — Prepare HCP Timeseries

- Loads the two hemisphere MAT files (~785 MB each)
- Parses `movie_timing.csv` → correct MAT offsets for each video segment
- Applies vicsompy-style train/test split:
  - **Train:** videos 1–4, 6–8, 10–13, 15–17 — z-scored per-run
  - **Test:** videos 5, 9, 14, 18 (82 TRs each) — z-scored per clip, concatenated → 328 TRs
- Projects BOLD onto LBOEs → design matrix X (T × 800)
- Extracts CIFTI grayordinate targets Y (T × 59412)

In [4]:
banner("SCRIPT 2 — Prepare HCP Timeseries")
t0 = time.time()

s2 = load_script("02_prep_hcp_timeseries.py")
s2.main()

elapsed = time.time() - t0
print(f"\n✓ Script 2 complete in {elapsed/60:.1f} min")


════════════════════════════════════════════════════════════
  SCRIPT 2 — Prepare HCP Timeseries
════════════════════════════════════════════════════════════
11:30:22  INFO  ============================================================
11:30:22  INFO  Script 2 — Prepare HCP timeseries
11:30:22  INFO  ============================================================
11:30:22  INFO  
Loading subsurfaces from Script 1 …
11:30:22  INFO    A1 — L eigenvectors: (77, 64), R eigenvectors: (66, 64)  (n_lboe=64)
11:30:22  INFO    V1 — L eigenvectors: (831, 200), R eigenvectors: (787, 200)  (n_lboe=200)
11:30:22  INFO  
Loading fMRI MAT files …
11:30:22  INFO    Loading notmean_left_Meanfile.mat …
11:30:25  INFO      Found key='Left_data', shape=(32492, 3655)
11:30:25  INFO      Final shape: (32492, 3655)  (verts × time)
11:30:25  INFO    Loading notmean_right_Meanfile.mat …
11:30:28  INFO      Found key='Right_data', shape=(32492, 3655)
11:30:28  INFO      Final shape: (32492, 3655)  (verts × time)
1

pixdim[1,2,3] should be non-zero; setting 0 dims to 1


11:30:30  WARNING  pixdim[1,2,3] should be non-zero; setting 0 dims to 1


pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1


11:30:30  INFO  pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1
11:30:30  INFO    Grayordinate vertices — left: 29696, right: 29716, total: 59412
11:30:30  INFO    Y_train: (2834, 59412)
11:30:30  INFO    Y_test : (328, 59412)
11:30:30  INFO  
Projecting BOLD onto LBOEs …
11:30:31  INFO    X_train: (2834, 528)  (Audio cols 0-127 [128 cols], Video cols 128-527 [400 cols])
11:30:31  INFO    X_test : (328, 528)
11:30:31  INFO  
Saving to /home/amin/Research/Representation/Movie/outputs/vicsompy_audiovisual/prep …
11:30:31  INFO    Saved: X_train.npy  Y_train.npy  X_test.npy  Y_test.npy  run_onsets.npy  band_sizes.npy
11:30:31  INFO  
Script 2 complete.

✓ Script 2 complete in 0.2 min


In [5]:
# ── Verify output shapes ──────────────────────────────────────────────────────
import pickle

PREP_DIR  = s2.PREP_DIR
CACHE_DIR = s2.CACHE_DIR

# Determine actual design-matrix column counts from the cached subsurfaces
# (n_lboe may be < 200 for small ROIs like A1)
with open(os.path.join(CACHE_DIR, "sub_a1.pkl"), "rb") as fh:
    _sub_a1 = pickle.load(fh)
with open(os.path.join(CACHE_DIR, "sub_v1.pkl"), "rb") as fh:
    _sub_v1 = pickle.load(fh)
n_audio_cols = _sub_a1.L_eigenvectors.shape[1] + _sub_a1.R_eigenvectors.shape[1]
n_video_cols = _sub_v1.L_eigenvectors.shape[1] + _sub_v1.R_eigenvectors.shape[1]
expected_x_cols = n_audio_cols + n_video_cols
print(f"Expected X cols: {expected_x_cols}  "
      f"(audio={n_audio_cols}, video={n_video_cols})")

arrays = {}
expected_shapes = {
    "X_train"   : (None, expected_x_cols),
    "Y_train"   : (None, 59412),
    "X_test"    : (328,  expected_x_cols),
    "Y_test"    : (328,  59412),
    "run_onsets": (4,),
}

print("\nLoaded arrays:")
for name, exp in expected_shapes.items():
    arr = np.load(os.path.join(PREP_DIR, f"{name}.npy"))
    arrays[name] = arr
    ok = all(e is None or arr.shape[i] == e for i, e in enumerate(exp))
    status = "✓" if ok else "✗ MISMATCH"
    print(f"  {name:12s}: {str(arr.shape):25s}  expected {str(exp):25s}  {status}")

band_sizes = np.load(os.path.join(PREP_DIR, "band_sizes.npy"))
print(f"\nband_sizes      : audio={band_sizes[0]}, video={band_sizes[1]}")
print(f"Run onsets      : {arrays['run_onsets'].tolist()}")
print(f"T_train         : {arrays['X_train'].shape[0]} TRs")
print(f"T_test          : {arrays['X_test'].shape[0]} TRs  (4 × 82)")
print(f"Design matrix   : audio cols 0-{n_audio_cols-1}, video cols {n_audio_cols}-{expected_x_cols-1}")

# Basic value sanity: z-scored data should have near-zero mean
print(f"\nY_train mean    : {arrays['Y_train'].mean():.4f}  (expect ≈0)")
print(f"Y_train std     : {arrays['Y_train'].std():.4f}")
print(f"X_train has NaN : {np.isnan(arrays['X_train']).any()}  (expect False)")
print(f"Y_train has NaN : {np.isnan(arrays['Y_train']).any()}  (expect False)")

Expected X cols: 528  (audio=128, video=400)

Loaded arrays:
  X_train     : (2834, 528)                expected (None, 528)                ✓
  Y_train     : (2834, 59412)              expected (None, 59412)              ✓
  X_test      : (328, 528)                 expected (328, 528)                 ✓
  Y_test      : (328, 59412)               expected (328, 59412)               ✓
  run_onsets  : (4,)                       expected (4,)                       ✓

band_sizes      : audio=128, video=400
Run onsets      : [0, 680, 1412, 2118]
T_train         : 2834 TRs
T_test          : 328 TRs  (4 × 82)
Design matrix   : audio cols 0-127, video cols 128-527

Y_train mean    : -0.0000  (expect ≈0)
Y_train std     : 1.0000
X_train has NaN : False  (expect False)
Y_train has NaN : False  (expect False)


---
## Script 3 — Fit Banded Ridge Regression

- Sets himalaya torch backend
- Builds `ColumnKernelizer` → `MultipleKernelRidgeCV` pipeline
  - Audio band: `slice(0, 400)` — bilateral A1 LBOEs
  - Video band: `slice(400, 800)` — bilateral V1 LBOEs
- Fits with leave-one-run-out CV (4 folds, `random_search`, 20 iterations)
- Evaluates on held-out 328-TR test set
- Computes variance partitioning:
  - `R2_audio`, `R2_video` — unique variance per modality
  - `Shared_R2 = R2_audio + R2_video − R2_full` — audiovisual integration

> **Note:** Wall-clock ~1–2 h depending on GPU availability.

In [6]:
banner("SCRIPT 3 — Fit Banded Ridge Regression")
t0 = time.time()

s3 = load_script("03_fit_banded_ridge.py")
s3.main()

elapsed = time.time() - t0
print(f"\n✓ Script 3 complete in {elapsed/60:.1f} min")


════════════════════════════════════════════════════════════
  SCRIPT 3 — Fit Banded Ridge Regression
════════════════════════════════════════════════════════════
11:30:32  INFO  ============================================================
11:30:32  INFO  Script 3 — Fit banded ridge regression
11:30:32  INFO    Backend       : torch  (vicsompy default)
11:30:32  INFO    Solver        : random_search  (vicsompy default)
11:30:32  INFO    Alphas        : logspace(1,20,20)
11:30:32  INFO    n_iter        : 20
11:30:32  INFO    Targets batch : 400
11:30:32  INFO  ============================================================
11:30:32  INFO  
Initialising himalaya backend: 'torch' …
11:30:33  INFO    Backend ready: <module 'himalaya.backend.torch' from '/home/amin/miniconda3/envs/vicsompy_av/lib/python3.10/site-packages/himalaya/backend/torch.py'>
11:30:33  INFO  Loading pre-processed arrays from /home/amin/Research/Representation/Movie/outputs/vicsompy_audiovisual/prep …
11:30:33  INFO    X

pixdim[1,2,3] should be non-zero; setting 0 dims to 1


11:36:15  WARNING  pixdim[1,2,3] should be non-zero; setting 0 dims to 1


pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1


11:36:15  INFO  pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1
11:36:15  INFO    R2_full: /home/amin/Research/Representation/Movie/outputs/vicsompy_audiovisual/R2_full.npy
11:36:15  INFO    R2_full: /home/amin/Research/Representation/Movie/outputs/vicsompy_audiovisual/cifti_maps/R2_full.dscalar.nii
11:36:15  INFO    R2_audio: /home/amin/Research/Representation/Movie/outputs/vicsompy_audiovisual/R2_audio.npy
11:36:15  INFO    R2_audio: /home/amin/Research/Representation/Movie/outputs/vicsompy_audiovisual/cifti_maps/R2_audio.dscalar.nii
11:36:15  INFO    R2_video: /home/amin/Research/Representation/Movie/outputs/vicsompy_audiovisual/R2_video.npy
11:36:15  INFO    R2_video: /home/amin/Research/Representation/Movie/outputs/vicsompy_audiovisual/cifti_maps/R2_video.dscalar.nii
11:36:15  INFO    Shared_R2: /home/amin/Research/Representation/Movie/outputs/vicsompy_audiovisual/Shared_R2.npy
11:36:15  INFO    Shared_R2: /home/amin/Research/Representation/Movie/outputs/vicsompy_a

In [7]:
# ── Verify output files and summarise variance maps ───────────────────────────
import nibabel as nib

OUTPUT_DIR = s3.OUTPUT_DIR
CIFTI_DIR  = s3.CIFTI_DIR

maps = {}
print("Variance maps (test-set R²):")
print(f"  {'Map':12s}  {'Shape':12s}  {'Mean':>8s}  {'Median':>8s}  {'Frac>0.01':>10s}")
print("  " + "-" * 58)

for name in ["R2_full", "R2_audio", "R2_video", "Shared_R2"]:
    npy_path   = os.path.join(OUTPUT_DIR, f"{name}.npy")
    cifti_path = os.path.join(CIFTI_DIR,  f"{name}.dscalar.nii")

    arr = np.load(npy_path)
    maps[name] = arr

    frac_pos = np.mean(arr > 0.01)
    print(f"  {name:12s}  {str(arr.shape):12s}  {np.nanmean(arr):8.4f}  "
          f"{np.nanmedian(arr):8.4f}  {frac_pos:10.1%}")

    # Check CIFTI file loads and has correct shape
    img = nib.load(cifti_path)
    assert img.shape == (1, 59412), f"Unexpected CIFTI shape: {img.shape}"

print()
print("CIFTI files:")
for name in ["R2_full", "R2_audio", "R2_video", "Shared_R2"]:
    p = os.path.join(CIFTI_DIR, f"{name}.dscalar.nii")
    size_mb = os.path.getsize(p) / 1e6
    print(f"  {p}  ({size_mb:.1f} MB)")

# Variance partition sanity: Shared = audio + video - full
shared_check = maps["R2_audio"] + maps["R2_video"] - maps["R2_full"]
max_diff = np.max(np.abs(shared_check - maps["Shared_R2"]))
print(f"\nShared_R2 formula check (max |diff|): {max_diff:.2e}  (expect ~0)")

Variance maps (test-set R²):
  Map           Shape             Mean    Median   Frac>0.01
  ----------------------------------------------------------
  R2_full       (59412,)        0.3580    0.3125       98.3%
  R2_audio      (59412,)        0.0753    0.0388       67.9%
  R2_video      (59412,)        0.2827    0.2288       94.7%
  Shared_R2     (59412,)       -0.0000    0.0000        0.0%

CIFTI files:
  /home/amin/Research/Representation/Movie/outputs/vicsompy_audiovisual/cifti_maps/R2_full.dscalar.nii  (0.6 MB)
  /home/amin/Research/Representation/Movie/outputs/vicsompy_audiovisual/cifti_maps/R2_audio.dscalar.nii  (0.6 MB)
  /home/amin/Research/Representation/Movie/outputs/vicsompy_audiovisual/cifti_maps/R2_video.dscalar.nii  (0.6 MB)
  /home/amin/Research/Representation/Movie/outputs/vicsompy_audiovisual/cifti_maps/Shared_R2.dscalar.nii  (0.6 MB)

Shared_R2 formula check (max |diff|): 0.00e+00  (expect ~0)


---
## Summary

All three scripts completed. Outputs are in:

```
/home/amin/Research/Representation/Movie/outputs/vicsompy_audiovisual/
├── subsurfaces/
│   ├── sub_a1.pkl          ← A1 Subsurface + n_lboe_a1 LBOEs/hem (≤200, capped by ROI size)
│   └── sub_v1.pkl          ← V1 Subsurface + 200 LBOEs/hem
├── prep/
│   ├── X_train.npy         ← (T_train, 2*n_lboe_a1 + 2*n_lboe_v1)
│   ├── Y_train.npy         ← (T_train, 59412)
│   ├── X_test.npy          ← (328, 2*n_lboe_a1 + 2*n_lboe_v1)
│   ├── Y_test.npy          ← (328, 59412)
│   ├── run_onsets.npy      ← (4,)
│   └── band_sizes.npy      ← [n_audio_cols, n_video_cols]
├── R2_full.npy             ← full model R²  (59412,)
├── R2_audio.npy            ← audio-unique R²
├── R2_video.npy            ← video-unique R²
├── Shared_R2.npy           ← audiovisual integration R²
└── cifti_maps/
    ├── R2_full.dscalar.nii
    ├── R2_audio.dscalar.nii
    ├── R2_video.dscalar.nii
    └── Shared_R2.dscalar.nii
```

**Note on A1 LBOE count:** A1 has ~70 vertices per hemisphere in the Glasser
parcellation — far fewer than the 200-eigenfunction target. `scipy.sparse.linalg.eigs`
requires `k < n`, so the actual count is capped to `min(200, n_L-2, n_R-2)`.
This is the mathematically correct behaviour and does not affect model validity;
it simply means the audio band uses fewer basis functions than V1.

Visualise in Connectome Workbench:
```bash
wb_view cifti_maps/R2_audio.dscalar.nii
```